In [1]:
import warnings
warnings.filterwarnings(action='ignore')
import requests
import folium

파이썬 데이터 처리 라이브러리 pandas를 설치하고 import 한다.

In [2]:
# !pip install pandas
import pandas as pd
# 파이썬의 딕셔너리 타입의 데이터를 pandas 데이터프레임으로 변환하기위해 import 한다.
from pandas import json_normalize

전국 또는 특정 지역의 스타벅스 매장 위치 정보를 가져와서 지도위에 시각화한다.

reqursts 모듈로 스타벅스 매장 위치 정보를 가져와서 딕셔너리 타입으로 변환한다.

In [3]:
targetSite = 'https://www.starbucks.co.kr/store/getStore.do?r=A357R6HM7P'
request = requests.post(targetSite, data = {
    'ins_lat': 37.4422,
    'ins_lng': 127.1367,
    'p_sido_cd': '',
    'p_gugun_cd': '0833',
    'in_biz_cd': '',
    'iend': 2500,
    'set_date': ''
})
storeList = request.json()

store = storeList['list']
for i in range(len(store)):
    print('{:4d}. {}: {}({}, {})'.format(i + 1, store[i]['s_name'], store[i]['doro_address'], store[i]['lat'], store[i]['lot']))

   1. 판교제2테크노밸리: 경기도 성남시 수정구 창업로 54 (시흥동)(37.41108510155048, 127.09463155093412)
   2. 위례중앙광장: 경기도 성남시 수정구 위례광장로 300 (창곡동) 130호(37.473711672971596, 127.1424768773468)
   3. 판교놀유니버스: 경기도 성남시 수정구 금토로 70 (금토동)(37.405652966500696, 127.08746383906066)
   4. 남위례역: 경기도 성남시 수정구 위례광장로 9-9 (창곡동)(37.4646, 127.14142)
   5. 성남모란삼성: 경기도 성남시 수정구 산성대로 81 (수진동)(37.43359712142669, 127.12801723528008)
   6. 판교위든타워: 경기도 성남시 수정구 금토로80번길 56 (금토동)(37.406853849128, 127.090099780526)
   7. 판교아이스퀘어: 경기도 성남시 수정구 창업로 18(시흥동)(37.41291102051743, 127.09848855739167)
   8. 성남태평: 경기도 성남시 수정구 수정로 149 (태평동) 1~2층(37.442561275669, 127.136588869764)
   9. 위례창곡: 경기도 성남시 수정구 위례동로 135 (창곡동)(37.472159, 127.150206)
  10. 수진역: 경기도 성남시 수정구 산성대로 223 (신흥동) 1층(37.4389903651714, 127.14224989005)
  11. 가천대학교: 경기도 성남시 수정구 성남대로 1342 (복정동)(37.44941, 127.1272)
  12. 성남이마트: 경기도 성남시 수정구 수정로 201 (태평1동) 신세계쉐덴(37.4442471, 127.14145710000002)
  13. 성남위례: 경기도 성남시 수정구 위례광장로 104 (창곡동)(37.472839, 127.141062)
  14. 신흥역: 경기도 성남시 수정구 산성대로 267 (신흥동)(37

json_normalize() 함수를 이용해서 json 타입의 데이터가 변환된 딕셔너리를 pandas 데이터프레임으로 변환한다.

In [4]:
starbucks_df = json_normalize(storeList, 'list')
# print(type(starbucks_df)) # <class 'pandas.DataFrame'>
# print(len(starbucks_df))
# print(starbucks_df.columns)
starbucks_df

,seq,sido_cd,sido_nm,gugun_cd,gugun_nm,code_order,view_yn,store_num,sido,gugun,...,t30,t36,t27,t29,t43,t48,z9999,t64,t66,p02
0,0,None,None,None,None,None,None,None,None,None,...,0,0,0,0,0,0,0,0,0,0
1,0,None,None,None,None,None,None,None,None,None,...,0,0,0,0,0,0,0,0,0,0
2,0,None,None,None,None,None,None,None,None,None,...,0,0,0,0,0,0,0,0,0,0
3,0,None,None,None,None,None,None,None,None,None,...,0,0,0,0,0,0,0,0,0,0
4,0,None,None,None,None,None,None,None,None,None,...,0,0,0,0,0,0,0,0,0,0
5,0,None,None,None,None,None,None,None,None,None,...,0,0,0,0,0,0,0,0,0,0
6,0,None,None,None,None,None,None,None,None,None,...,0,0,0,0,0,0,0,0,0,0
7,0,None,None,None,None,None,None,None,None,None,...,0,0,0,0,0,0,0,0,0,0
8,0,None,None,None,None,None,None,None,None,None,...,0,0,0,0,0,0,0,0,0,0
9,0,None,None,None,None,None,None,None,None,None,...,0,0,0,0,0,0,0,0,0,0


딕셔너리를 데이터프레임으로 변환하니 경기도 성남시 수정구에 있는 스타벅스 지점은 14개고 지점당 137개의 컬럼으로 정보를 표시하고 있다.  
작업에 필요한 컬럼 몇 가지를 선택해서 지도에 마커를 표시할 때 사용할 데이터프레임으로 만든다.

s_name => 지점이름  
sido_code => 시도코드  
sido_name => 시도이름  
gugun_code => 구군코드  
gugun_name => 구군이름  
doro_address => 도로명주소  
lat => 위도  
lot => 경도

In [5]:
starbucks_df_map = starbucks_df[
    ['s_name', 'sido_code', 'sido_name', 'gugun_code', 'gugun_name', 'doro_address', 'lat', 'lot']
]
del starbucks_df
starbucks_df_map

,s_name,sido_code,sido_name,gugun_code,gugun_name,doro_address,lat,lot
0,판교제2테크노밸리,08,경기,0833,성남시 수정구,경기도 성남시 수정구 창업로 54 (시흥동),37.41108510155048,127.09463155093412
1,위례중앙광장,08,경기,0833,성남시 수정구,경기도 성남시 수정구 위례광장로 300 (창곡동) 130호,37.473711672971596,127.1424768773468
2,판교놀유니버스,08,경기,0833,성남시 수정구,경기도 성남시 수정구 금토로 70 (금토동),37.405652966500696,127.08746383906066
3,남위례역,08,경기,0833,성남시 수정구,경기도 성남시 수정구 위례광장로 9-9 (창곡동),37.4646,127.14142
4,성남모란삼성,08,경기,0833,성남시 수정구,경기도 성남시 수정구 산성대로 81 (수진동),37.43359712142669,127.12801723528008
5,판교위든타워,08,경기,0833,성남시 수정구,경기도 성남시 수정구 금토로80번길 56 (금토동),37.406853849128,127.090099780526
6,판교아이스퀘어,08,경기,0833,성남시 수정구,경기도 성남시 수정구 창업로 18(시흥동),37.41291102051743,127.09848855739167
7,성남태평,08,경기,0833,성남시 수정구,경기도 성남시 수정구 수정로 149 (태평동) 1~2층,37.442561275669,127.136588869764
8,위례창곡,08,경기,0833,성남시 수정구,경기도 성남시 수정구 위례동로 135 (창곡동),37.472159,127.150206
9,수진역,08,경기,0833,성남시 수정구,경기도 성남시 수정구 산성대로 223 (신흥동) 1층,37.4389903651714,127.14224989005


dtypes 속성으로 데이터프레임을 구성하는 컬럼들의 자료형을 확인할 수 있다.  
pandas는 예전에 문자열 데이터를 str이 아니라 object로 사용한 적 있다.

In [7]:
starbucks_df_map.dtypes

s_name          str
sido_code       str
sido_name       str
gugun_code      str
gugun_name      str
doro_address    str
lat             str
lot             str
dtype: object

info() 메소드를 사용해도 데이터프레임을 구성하는 컬럼들의 자료형을 확인할 수 있다.

In [8]:
starbucks_df_map.info()

<class 'pandas.DataFrame'>
RangeIndex: 14 entries, 0 to 13
Data columns (total 8 columns):
 #   Column        Non-Null Count  Dtype
---  ------        --------------  -----
 0   s_name        14 non-null     str  
 1   sido_code     14 non-null     str  
 2   sido_name     14 non-null     str  
 3   gugun_code    14 non-null     str  
 4   gugun_name    14 non-null     str  
 5   doro_address  14 non-null     str  
 6   lat           14 non-null     str  
 7   lot           14 non-null     str  
dtypes: str(8)
memory usage: 1.0 KB


astype() 메소드로 위도, 경도 데이터의 타입을 str에서 float로 변경한다.

In [9]:
starbucks_df_map['lat'] = starbucks_df_map['lat'].astype(float)
starbucks_df_map['lot'] = starbucks_df_map['lot'].astype(float)
starbucks_df_map.info()

<class 'pandas.DataFrame'>
RangeIndex: 14 entries, 0 to 13
Data columns (total 8 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   s_name        14 non-null     str    
 1   sido_code     14 non-null     str    
 2   sido_name     14 non-null     str    
 3   gugun_code    14 non-null     str    
 4   gugun_name    14 non-null     str    
 5   doro_address  14 non-null     str    
 6   lat           14 non-null     float64
 7   lot           14 non-null     float64
dtypes: float64(2), str(6)
memory usage: 1.0 KB


folium 모듈을 사용해서 지도를 만들고 지도위에 스타벅스 매장 위치를 마커로 표시한다.

In [10]:
starbucks_df_map[starbucks_df_map['s_name'] == '성남태평']

,s_name,sido_code,sido_name,gugun_code,gugun_name,doro_address,lat,lot
7,성남태평,08,경기,0833,성남시 수정구,경기도 성남시 수정구 수정로 149 (태평동) 1~2층,37.442561,127.136589


iterrows() 메소드는 데이터프레임에 저장된 데이터의 인덱스와 데이터를 튜플 형태로 리턴한다.

In [12]:
for index, data in starbucks_df_map.iterrows():
    print(index, data)

0 s_name                         판교제2테크노밸리
sido_code                             08
sido_name                             경기
gugun_code                          0833
gugun_name                       성남시 수정구
doro_address    경기도 성남시 수정구 창업로 54 (시흥동)
lat                            37.411085
lot                           127.094632
Name: 0, dtype: object
1 s_name                                    위례중앙광장
sido_code                                     08
sido_name                                     경기
gugun_code                                  0833
gugun_name                               성남시 수정구
doro_address    경기도 성남시 수정구 위례광장로 300 (창곡동) 130호
lat                                    37.473712
lot                                   127.142477
Name: 1, dtype: object
2 s_name                           판교놀유니버스
sido_code                             08
sido_name                             경기
gugun_code                          0833
gugun_name                       성남시 수정구
doro_address    경기도 성남시

In [14]:
foliumMap = folium.Map(location=[starbucks_df_map['lat'].mean(), starbucks_df_map['lot'].mean()], zoom_start=13)

for index, data in starbucks_df_map.iterrows():
    popup = folium.Popup('{}. {} - {}'.format(index + 1, data['s_name'], data['doro_address']), max_width=300)
    folium.Marker(location=[data['lat'], data['lot']], popup=popup).add_to(foliumMap)

foliumMap.save('./output/starbucksMap1.html')
foliumMap

서울시 강남구

In [15]:
targetSite = 'https://www.starbucks.co.kr/store/getStore.do?r=A357R6HM7P'
request = requests.post(targetSite, data = {
    'ins_lat': 37.4422,
    'ins_lng': 127.1367,
    'p_sido_cd': '',
    'p_gugun_cd': '0101',
    'in_biz_cd': '',
    'iend': 2500,
    'set_date': ''
})
storeList = request.json()

starbucks_df = json_normalize(storeList, 'list')
starbucks_df_map = starbucks_df[['s_name', 'sido_code', 'sido_name', 'gugun_code', 'gugun_name', 'doro_address', 'lat', 'lot']]
starbucks_df_map['lat'] = starbucks_df_map['lat'].astype(float)
starbucks_df_map['lot'] = starbucks_df_map['lot'].astype(float)

foliumMap = folium.Map(location=[starbucks_df_map['lat'].mean(), starbucks_df_map['lot'].mean()], zoom_start=13)
for index, data in starbucks_df_map.iterrows():
    popup = folium.Popup('{}. {} - {}'.format(index + 1, data['s_name'], data['doro_address']), max_width=300)
    folium.Marker(location=[data['lat'], data['lot']], popup=popup).add_to(foliumMap)
foliumMap.save('./output/starbucksMap2.html')
foliumMap

In [20]:
targetSite = 'https://www.starbucks.co.kr/store/getStore.do?r=A357R6HM7P'
request = requests.post(targetSite, data = {
    'ins_lat': 37.4422,
    'ins_lng': 127.1367,
    'p_sido_cd': '01',
    'p_gugun_cd': '',
    'in_biz_cd': '',
    'iend': 2500,
    'set_date': ''
})
storeList = request.json()

starbucks_df = json_normalize(storeList, 'list')
starbucks_df_map = starbucks_df[['s_name', 'sido_code', 'sido_name', 'gugun_code', 'gugun_name', 'doro_address', 'lat', 'lot']]
starbucks_df_map['lat'] = starbucks_df_map['lat'].astype(float)
starbucks_df_map['lot'] = starbucks_df_map['lot'].astype(float)

foliumMap = folium.Map(location=[starbucks_df_map['lat'].mean(), starbucks_df_map['lot'].mean()], zoom_start=12)
for index, data in starbucks_df_map.iterrows():
    popup = folium.Popup('{}. {} - {}'.format(index + 1, data['s_name'], data['doro_address']), max_width=300)
    folium.Marker(location=[data['lat'], data['lot']], popup=popup).add_to(foliumMap)
foliumMap.save('./output/starbucksMap3.html')
foliumMap

In [21]:
targetSite = 'https://www.starbucks.co.kr/store/getStore.do?r=A357R6HM7P'
request = requests.post(targetSite, data = {
    'ins_lat': 37.4422,
    'ins_lng': 127.1367,
    'p_sido_cd': '03',
    'p_gugun_cd': '',
    'in_biz_cd': '',
    'iend': 2500,
    'set_date': ''
})
storeList = request.json()

starbucks_df = json_normalize(storeList, 'list')
starbucks_df_map = starbucks_df[['s_name', 'sido_code', 'sido_name', 'gugun_code', 'gugun_name', 'doro_address', 'lat', 'lot']]
starbucks_df_map['lat'] = starbucks_df_map['lat'].astype(float)
starbucks_df_map['lot'] = starbucks_df_map['lot'].astype(float)

foliumMap = folium.Map(location=[starbucks_df_map['lat'].mean(), starbucks_df_map['lot'].mean()], zoom_start=12)
for index, data in starbucks_df_map.iterrows():
    popup = folium.Popup('{}. {} - {}'.format(index + 1, data['s_name'], data['doro_address']), max_width=300)
    folium.Marker(location=[data['lat'], data['lot']], popup=popup).add_to(foliumMap)
foliumMap.save('./output/starbucksMap4.html')
foliumMap

In [24]:
targetSite = 'https://www.starbucks.co.kr/store/getStore.do?r=A357R6HM7P'
request = requests.post(targetSite, data = {
    'ins_lat': 37.4422,
    'ins_lng': 127.1367,
    'p_sido_cd': '16',
    'p_gugun_cd': '',
    'in_biz_cd': '',
    'iend': 2500,
    'set_date': ''
})
storeList = request.json()

starbucks_df = json_normalize(storeList, 'list')
starbucks_df_map = starbucks_df[['s_name', 'sido_code', 'sido_name', 'gugun_code', 'gugun_name', 'doro_address', 'lat', 'lot']]
starbucks_df_map['lat'] = starbucks_df_map['lat'].astype(float)
starbucks_df_map['lot'] = starbucks_df_map['lot'].astype(float)

foliumMap = folium.Map(location=[starbucks_df_map['lat'].mean(), starbucks_df_map['lot'].mean()], zoom_start=11)
for index, data in starbucks_df_map.iterrows():
    popup = folium.Popup('{}. {} - {}'.format(index + 1, data['s_name'], data['doro_address']), max_width=300)
    folium.Marker(location=[data['lat'], data['lot']], popup=popup).add_to(foliumMap)
foliumMap.save('./output/starbucksMap5.html')
foliumMap

In [26]:
targetSite = 'https://www.starbucks.co.kr/store/getStore.do?r=A357R6HM7P'
request = requests.post(targetSite, data = {
    'ins_lat': 37.4422,
    'ins_lng': 127.1367,
    'p_sido_cd': '',
    'p_gugun_cd': '',
    'in_biz_cd': '',
    'iend': 2500,
    'set_date': ''
})
storeList = request.json()

starbucks_df = json_normalize(storeList, 'list')
starbucks_df_map = starbucks_df[['s_name', 'sido_code', 'sido_name', 'gugun_code', 'gugun_name', 'doro_address', 'lat', 'lot']]
starbucks_df_map['lat'] = starbucks_df_map['lat'].astype(float)
starbucks_df_map['lot'] = starbucks_df_map['lot'].astype(float)

foliumMap = folium.Map(location=[starbucks_df_map['lat'].mean(), starbucks_df_map['lot'].mean()], zoom_start=7)
for index, data in starbucks_df_map.iterrows():
    popup = folium.Popup('{}. {} - {}'.format(index + 1, data['s_name'], data['doro_address']), max_width=300)
    folium.Marker(location=[data['lat'], data['lot']], popup=popup).add_to(foliumMap)
foliumMap.save('./output/starbucksMap5.html')
foliumMap